# Medical Chatbot RAG — Hugging Face Local LLM

Workflow: PDF loading → cleaning → chunking → Hugging Face embeddings → Pinecone → retrieval → local Hugging Face answer generation.

No OpenAI API key or OpenAI credits are required for answer generation.


## 0. Install compatible packages
Run this once in a fresh environment/kernel, then restart the kernel if Jupyter asks you to.

In [1]:
%pip install -U "langchain==0.3.27" "langchain-community==0.3.27" "langchain-text-splitters==0.3.9" "langchain-huggingface==0.3.1" "langchain-pinecone==0.2.12" "pinecone==7.3.0" "pypdf>=5.0.0" "sentence-transformers>=3.0.0" "python-dotenv>=1.0.1" "transformers>=4.45,<6" "accelerate>=0.30" "sentencepiece>=0.2" "fonttools>=4.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

print("Working directory:", Path.cwd())

Working directory: c:\medical_chatbot\medical-help-chatbot\research


In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

## 1. Load PDF files

In [4]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# ============================================================
# PERMANENT PROJECT PATH SETUP
# ============================================================

# Project root:
# C:\medical_chatbot\medical-help-chatbot

PROJECT_ROOT = Path.cwd().parent

# If notebook is already opened from project root, use it directly
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()

# If running from research/ folder
elif (Path.cwd().parent / "data").exists():
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

print("Current notebook directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nPDF data directory:")
print(DATA_DIR)

# ============================================================
# CHECK DATA FOLDER
# ============================================================

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"""
Data folder not found.

Expected location:
{DATA_DIR}

Please make sure your project has:
{PROJECT_ROOT}\\data\\
"""
    )

# ============================================================
# FIND PDF FILES
# ============================================================

pdf_files = list(DATA_DIR.rglob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found inside:\n{DATA_DIR}"
    )

print("\nPDF files found:")

for pdf in pdf_files:
    print(" -", pdf)

# ============================================================
# LOAD PDFs
# ============================================================

loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    silent_errors=False
)

extracted_data = loader.load()

print(f"\nSuccessfully loaded {len(extracted_data)} document pages.")

Current notebook directory:
c:\medical_chatbot\medical-help-chatbot\research

Project root:
c:\medical_chatbot\medical-help-chatbot

PDF data directory:
c:\medical_chatbot\medical-help-chatbot\data

PDF files found:
 - c:\medical_chatbot\medical-help-chatbot\data\Medical_book.pdf


100%|██████████| 1/1 [00:33<00:00, 33.82s/it]


Successfully loaded 637 document pages.


In [5]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# ============================================================
# PERMANENT PROJECT PATH SETUP
# ============================================================

# Project root:
# C:\medical_chatbot\medical-help-chatbot

PROJECT_ROOT = Path.cwd().parent

# If notebook is already opened from project root, use it directly
if (Path.cwd() / "data").exists():
    PROJECT_ROOT = Path.cwd()

# If running from research/ folder
elif (Path.cwd().parent / "data").exists():
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

print("Current notebook directory:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nPDF data directory:")
print(DATA_DIR)

# ============================================================
# CHECK DATA FOLDER
# ============================================================

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"""
Data folder not found.

Expected location:
{DATA_DIR}

Please make sure your project has:
{PROJECT_ROOT}\\data\\
"""
    )

# ============================================================
# FIND PDF FILES
# ============================================================

pdf_files = list(DATA_DIR.rglob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found inside:\n{DATA_DIR}"
    )

print("\nPDF files found:")

for pdf in pdf_files:
    print(" -", pdf)

# ============================================================
# LOAD PDFs
# ============================================================

loader = DirectoryLoader(
    str(DATA_DIR),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    silent_errors=False
)

extracted_data = loader.load()

print(f"\nSuccessfully loaded {len(extracted_data)} document pages.")

Current notebook directory:
c:\medical_chatbot\medical-help-chatbot\research

Project root:
c:\medical_chatbot\medical-help-chatbot

PDF data directory:
c:\medical_chatbot\medical-help-chatbot\data

PDF files found:
 - c:\medical_chatbot\medical-help-chatbot\data\Medical_book.pdf


100%|██████████| 1/1 [00:34<00:00, 34.48s/it]


Successfully loaded 637 document pages.


In [6]:
extracted_data[:2]

[Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'c:\\medical_chatbot\\medical-help-chatbot\\data\\Medical_book.pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'c:\\medical_chatbot\\medical-help-chatbot\\data\\Medical_book.pdf', 'total_pages': 637, 'page': 1, 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')]

In [7]:
len(extracted_data)

637

## 2. Keep only useful metadata

In [8]:
def filter_to_minimal_docs(docs: list[Document]) -> list[Document]:
    return [Document(page_content=doc.page_content, metadata={"source": doc.metadata.get("source")}) for doc in docs]

minimal_docs = filter_to_minimal_docs(extracted_data)
minimal_docs[:2]

[Document(metadata={'source': 'c:\\medical_chatbot\\medical-help-chatbot\\data\\Medical_book.pdf'}, page_content=''),
 Document(metadata={'source': 'c:\\medical_chatbot\\medical-help-chatbot\\data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')]

## 3. Split documents into chunks

In [9]:
def text_split(docs: list[Document]) -> list[Document]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20, length_function=len)
    return text_splitter.split_documents(docs)

texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5860


In [10]:
texts_chunk[:2]

[Document(metadata={'source': 'c:\\medical_chatbot\\medical-help-chatbot\\data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'c:\\medical_chatbot\\medical-help-chatbot\\data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1')]

## 4. Hugging Face embeddings (no OpenAI embeddings required)

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", embedding_device)
embedding = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": embedding_device},
    encode_kwargs={"normalize_embeddings": True}
)
vector = embedding.embed_query("Hello world")
print("Vector length:", len(vector))


Embedding device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector length: 384


## 5. Environment variables
Create a `.env` file in the project root with only the Pinecone key:

```text
PINECONE_API_KEY=your_pinecone_key
```

No OpenAI API key is needed. Do not paste keys directly into the notebook or commit `.env` to GitHub.


In [12]:
from dotenv import load_dotenv
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("Missing PINECONE_API_KEY. Add it to the project-root .env file and restart the kernel.")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
print("Pinecone API key loaded successfully (value hidden).")


Pinecone API key loaded successfully (value hidden).


## 6. Connect to Pinecone

In [13]:
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "medical-chatbot"
embedding_dimension = len(vector)
if not pc.has_index(index_name):
    pc.create_index(name=index_name, dimension=embedding_dimension, metric="cosine", spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    print(f"Created Pinecone index: {index_name}")
else:
    print(f"Using existing Pinecone index: {index_name}")
index = pc.Index(index_name)
print(index.describe_index_stats())

Using existing Pinecone index: medical-chatbot
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 18580}},
 'total_vector_count': 18580,
 'vector_type': 'dense'}


## 7. Create/connect the LangChain Pinecone vector store
The old notebook uploaded documents and then immediately opened the same index again. Re-running it can create duplicate vectors. Use `BUILD_INDEX=True` only when you intentionally upload the current chunks.

In [14]:
import sys
print(sys.executable)
print(sys.version)

c:\medical_chatbot\medical-help-chatbot\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [15]:
%pip install "langchain-pinecone==0.2.12"


Note: you may need to restart the kernel to use updated packages.


In [16]:
from langchain_pinecone import PineconeVectorStore

# Reuse the existing Pinecone index. Upload chunks only if the index is empty.
stats = index.describe_index_stats()
total_vectors = getattr(stats, "total_vector_count", 0)
print("Existing vectors in Pinecone:", total_vectors)

if total_vectors == 0:
    docsearch = PineconeVectorStore.from_documents(
        documents=texts_chunk,
        embedding=embedding,
        index_name=index_name
    )
    print(f"Uploaded {len(texts_chunk)} chunks to Pinecone.")
else:
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embedding
    )
    print("Connected to the existing Pinecone vector store.")


c:\medical_chatbot\medical-help-chatbot\.venv\Lib\site-packages\langchain_pinecone\__init__.py:3: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_pinecone.vectorstores import Pinecone, PineconeVectorStore


Existing vectors in Pinecone: 18580
Connected to the existing Pinecone vector store.


## 8. Add extra data to the existing index

In [17]:
dswith = Document(page_content="dswithbappy is a youtube channel that provides tutorials on various topics.", metadata={"source":"Youtube"})
ADD_EXTRA_DOCUMENT = False
if ADD_EXTRA_DOCUMENT:
    docsearch.add_documents(documents=[dswith])
    print("Extra document added.")

## 9. Retriever

In [18]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})
retrieved_docs = retriever.invoke("What is Acne?")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Document {i} | {doc.metadata.get('source')} ---")
    print(doc.page_content[:800])
    print()

--- Document 1 | data\Medical_book.pdf ---
GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26

--- Document 2 | c:\medical_chatbot\medical-help-chatbot\data\Medical_book.pdf ---
GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26

--- Document 3 | c:\medical_chatbot\medical-help-chatbot\data\Medical_book.pdf ---
GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26



## 10. Local Hugging Face language model
This notebook uses a local Hugging Face model, so OpenAI credits/API keys are not required.


In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LOCAL_LLM = "google/flan-t5-small"
llm_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Local LLM:", LOCAL_LLM)
print("Device:", llm_device)

tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM)
model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL_LLM)
model = model.to(llm_device)
model.eval()
print("Local Hugging Face model loaded successfully.")


Local LLM: google/flan-t5-small
Device: cpu


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

c:\medical_chatbot\medical-help-chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

c:\medical_chatbot\medical-help-chatbot\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:05<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Local Hugging Face model loaded successfully.


## 11. Build the local RAG answer function


In [20]:
def generate_local_answer(question: str, context: str) -> str:
    prompt = (
        "You are a medical information assistant. Answer only from the provided medical-book context. "
        "If the answer is not in the context, say: I could not find this information in the provided medical book. "
        "Do not invent medical facts. Keep the answer concise and clear.\n\n"
        f"Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(llm_device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

def ask(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    answer = generate_local_answer(question, context)
    response = {"answer": answer, "context": docs}
    print(answer)
    return response

print("Local RAG chain ready.")


Local RAG chain ready.


## 12. Ask questions

In [21]:
# OpenAI is intentionally not used in this notebook.
print("OpenAI API: not used — local Hugging Face generation is enabled.")


OpenAI API: not used — local Hugging Face generation is enabled.


In [22]:
retrieved_docs = retriever.invoke(
    "What is Acromegaly and gigantism?"
)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:1000])


--- Document 1 ---
Whitehouse Station, NJ: Merck Research Laboratories,
1997.
Larsen, D. E., ed. Mayo Clinic Family Health Book.New York:
William Morrow and Co., Inc., 1996.
John T. Lohr, PhD
Acromegaly and gigantism
Definition
Acromegaly is a disorder in which the abnormal
release of a particular chemical from the pituitary gland
in the brain causes increased growth in bone and soft tis-
sue, as well as a variety of other disturbances throughout
the body. This chemical released from the pituitary gland

--- Document 2 ---
Whitehouse Station, NJ: Merck Research Laboratories,
1997.
Larsen, D. E., ed. Mayo Clinic Family Health Book.New York:
William Morrow and Co., Inc., 1996.
John T. Lohr, PhD
Acromegaly and gigantism
Definition
Acromegaly is a disorder in which the abnormal
release of a particular chemical from the pituitary gland
in the brain causes increased growth in bone and soft tis-
sue, as well as a variety of other disturbances throughout
the body. This chemical released from 

In [23]:
# OpenAI is intentionally not used in this notebook.
print("OpenAI API: not used — local Hugging Face generation is enabled.")


OpenAI API: not used — local Hugging Face generation is enabled.


In [24]:
# OpenAI is intentionally not used in this notebook.
print("OpenAI API: not used — local Hugging Face generation is enabled.")


OpenAI API: not used — local Hugging Face generation is enabled.


In [25]:
response = ask("What is Acromegaly and gigantism?")


disorder in which the abnormal release of a particular chemical from the pituitary gland in the brain causes increased growth in bone and soft tis- sue, as well as a variety of other disturbances throughout the body.


In [26]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")


PyTorch version: 2.14.1+cpu
CUDA available: False
Using CPU


In [27]:
import sys
import torch

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable: c:\medical_chatbot\medical-help-chatbot\.venv\Scripts\python.exe
PyTorch: 2.14.1+cpu
CUDA available: False


In [28]:
response = ask("What is Acromegaly and gigantism?")


disorder in which the abnormal release of a particular chemical from the pituitary gland in the brain causes increased growth in bone and soft tis- sue, as well as a variety of other disturbances throughout the body.


In [29]:
response = ask("What is Acne?")

GALE ENCYCLOPEDIA OF MEDICINE 226


In [30]:
response = ask("What is the Treatment of Acne?")

Treatment for mild noninflamma- tory acne consists of reducing the formation of new comedones with topical tretinoin, benzoyl peroxide, ada-palene, or salicylic acid.


In [31]:
response = ask("what dswithbappy?")

a youtube channel


## Notes for rerunning
1. Keep PDFs inside `data/`.
2. Keep API keys in `.env`.
3. Run installation once per environment.
4. Use `BUILD_INDEX=True` only for the first upload or an intentional rebuild.
5. Then set `BUILD_INDEX=False` to avoid duplicate vectors.
6. Hugging Face handles embeddings locally; the OpenAI key is used only for chat generation.